# Adversarial Training with FGSM

This notebook demonstrates **adversarial training**, a defense mechanism against adversarial attacks.

## Overview
- **Adversarial Training**: Training on both clean and adversarial examples to improve model robustness
- **FGSM Attack**: Used during training to generate adversarial examples
- **Goal**: Build a model that maintains accuracy on both clean and adversarial data

## Key Concept
Instead of training only on clean images, we:
1. Generate adversarial examples using FGSM
2. Train the model on both clean and adversarial examples
3. This forces the model to learn robust decision boundaries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

print("Libraries imported successfully")

## 1. Data Preparation

Load and prepare the MNIST dataset for adversarial training.

In [ ]:
# Define data transformations
transform = transforms.Compose([
    transforms.ToTensor()
])

# Load MNIST training dataset
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Load MNIST test dataset
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

100%|██████████| 9.91M/9.91M [00:30<00:00, 326kB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 94.4kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.59MB/s]


## 2. Model Architecture

Define a CNN model for MNIST classification. The same architecture will be used for both normal and adversarial training.

- **Conv Layer 1**: 1 input channel → 32 filters (3×3 kernel)
- **Conv Layer 2**: 32 channels → 64 filters (3×3 kernel)
- **Fully Connected Layer 1**: 36864 → 128 neurons
- **Fully Connected Layer 2**: 128 → 10 output classes

In [10]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)

        self.fc1 = nn.Linear(36864, 128)
        self.fc2 = nn.Linear(128, 10)

        self.relu = nn.ReLU()

    def forward(self, x):

        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))

        x = torch.flatten(x, 1)

        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [ ]:
# Instantiate the model
model = CNN()
print(model)

### Initialize Training Configuration

Set up the loss function, optimizer, device, and hyperparameters for adversarial training.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epsilon = 0.25  # Perturbation magnitude for FGSM
epochs = 3

print(f"Using device: {device}")
print(f"Epsilon: {epsilon}")
print(f"Epochs: {epochs}")

## 3. FGSM Attack Function

Define the function to generate adversarial examples using the Fast Gradient Sign Method.

In [ ]:
def fgsm_attack(images, epsilon, data_grad):
    """
    Generate adversarial examples using FGSM.
    
    Args:
        images: Original input images
        epsilon: Magnitude of perturbation
        data_grad: Gradient of loss with respect to images
    
    Returns:
        Adversarial images clipped to valid pixel range [0, 1]
    """
    # Compute perturbation: epsilon * sign(gradient)
    sign_grad = data_grad.sign()
    
    # Add perturbation to original images
    perturbed_image = images + epsilon * sign_grad
    
    # Clip to valid pixel range [0, 1]
    perturbed_image = torch.clamp(perturbed_image, 0, 1)
    
    return perturbed_image

In [14]:
for epoch in range(epochs):

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images.requires_grad = True

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        data_grad = images.grad.data

        adv_images = fgsm_attack(images, epsilon, data_grad)

        adv_outputs = model(adv_images)

        adv_loss = criterion(adv_outputs, labels)

        optimizer.zero_grad()

        adv_loss.backward()

        optimizer.step()

        running_loss += adv_loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.7929
Epoch 2, Loss: 0.3670
Epoch 3, Loss: 0.3085


## 5. Evaluate on Clean Data

Test the adversarially trained model on clean (unperturbed) test data to measure accuracy on normal images.

In [15]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Clean Test Accuracy: {accuracy:.2f}%")

Clean Test Accuracy: 96.31%


## 6. Evaluate Robustness Against FGSM Attack

Test the adversarially trained model against FGSM attacks to measure how well it resists adversarial perturbations.

In [ ]:
# Test accuracy under FGSM attack
correct = 0
total = 0

epsilon = 0.25

model.eval()

for images, labels in test_loader:

    images = images.to(device)
    labels = labels.to(device)

    images.requires_grad = True

    outputs = model(images)
    loss = criterion(outputs, labels)

    model.zero_grad()
    loss.backward()

    data_grad = images.grad.data

    adv_images = fgsm_attack(images, epsilon, data_grad)

    adv_outputs = model(adv_images)

    _, predicted = torch.max(adv_outputs, 1)

    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"FGSM Attack Accuracy: {accuracy:.2f}%")

FGSM Attack Accuracy: 92.89%


## 7. Summary

**Key Results:**
- The adversarially trained model achieves good accuracy on **both clean data** and **adversarial examples**
- This demonstrates that adversarial training effectively improves model robustness
- The trade-off between clean accuracy and adversarial robustness is typically worth it for security-critical applications

**Why This Works:**
1. During training, the model encounters both clean and adversarial examples
2. This teaches the model to learn decision boundaries that are harder to cross with gradient-based attacks
3. The model develops more robust features that generalize better to perturbations

**Next Steps:**
- Try different epsilon values to see how robustness varies
- Compare with other defense mechanisms
- Test on different datasets
- Experiment with stronger attacks like PGD (Projected Gradient Descent)